In [25]:
import os
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import VotingRegressor,RandomForestRegressor, BaggingRegressor,StackingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer,make_column_selector
from sklearn.impute import SimpleImputer
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/BigMarketSale/')

In [26]:
train = pd.read_csv("train_v9rqX0R.csv")
test = pd.read_csv("test_AbJTz2l.csv")

In [27]:
train

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.300,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.500,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.200,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.930,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,FDF22,6.865,Low Fat,0.056783,Snack Foods,214.5218,OUT013,1987,High,Tier 3,Supermarket Type1,2778.3834
8519,FDS36,8.380,Regular,0.046982,Baking Goods,108.1570,OUT045,2002,NaN,Tier 2,Supermarket Type1,549.2850
8520,NCJ29,10.600,Low Fat,0.035186,Health and Hygiene,85.1224,OUT035,2004,Small,Tier 2,Supermarket Type1,1193.1136
8521,FDN46,7.210,Regular,0.145221,Snack Foods,103.1332,OUT018,2009,Medium,Tier 3,Supermarket Type2,1845.5976


In [ ]:
train.groupby['Item_Identifier']['Item_Weight'] = train.groupby['Item_Identifier']['Item_Weight'].mode()

In [28]:
train.isnull().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [29]:
ohe = OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore').set_output(transform='pandas')
SI = SimpleImputer(strategy='mean').set_output(transform='pandas')
SI_F = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
data_pipe = Pipeline([('SI_F',SI_F),('OHE',ohe)])

In [30]:
transformer = ColumnTransformer([('SI',SI,['Item_Weight']),
                                ('PIPE',data_pipe,make_column_selector(dtype_include=object))
                                 ],
                               verbose_feature_names_out=False,remainder='passthrough')

In [31]:
X, y = train.drop('Item_Outlet_Sales', axis = 1), train['Item_Outlet_Sales']

In [37]:
xgbm = XGBRegressor(random_state = 26)
lgbm = LGBMRegressor(random_state = 26)
cat = CatBoostRegressor(random_state = 26)
rf = RandomForestRegressor(random_state=26)
stack = StackingRegressor([('XGB',xgbm),('LGBM',lgbm),('CAT',cat)],final_estimator=rf)

In [39]:
kfolds = KFold(n_splits = 5, shuffle = True, random_state = 26)
pipe_stack = Pipeline([('OHE', transformer), ('stack',stack)])

params = {'stack__XGB__max_depth': [4, 6,10],
          'stack__XGB__learning_rate': [0.05, 0.2,0.3],
          'stack__XGB__n_estimators': [100, 150,200],
         
          'stack__LGBM__max_depth': [4,6,10],
          'stack__LGBM__learning_rate': [0.05, 0.1,0.3],
          'stack__LGBM__n_estimators': [75, 150,200],
          
          'stack__CAT__max_depth': [6,8,10],
          'stack__CAT__learning_rate': [0.05, 0.1, 0.2],
          'stack__CAT__n_estimators': [75,100,200],
          
          'stack__final_estimator__max_depth' : [4,6],
          'stack__final_estimator__n_estimators' : [75,100,150]
         }

gcv = RandomizedSearchCV(estimator= pipe_stack, cv = kfolds, n_jobs = -1, param_distributions=params, verbose = 2,scoring='neg_root_mean_squared_error')
gcv.fit(X, y)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.099025 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 815
[LightGBM] [Info] Number of data points in the train set: 6818, number of used features: 39
[LightGBM] [Info] Start training from score 2183.057911
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.272432 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 820
[LightGBM] [Info] Number of data points in the train set: 6819, number of used features: 39
[LightGBM] [Info] Start training from score 2189.804680
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.247659 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 819
[LightGBM] [Info] Number of data points in the train set: 6818, number of used features: 39
[LightGBM] [Info] Start training from score 2180.679468
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.239904 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 815
[LightGBM] [Info] Number of data points in the train set: 6818, number of used features: 39
[LightGBM] [Info] Start training from score 2183.057911
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

97:	learn: 1003.4321333	total: 4.93s	remaining: 101ms
98:	learn: 1002.8920289	total: 4.99s	remaining: 50.4ms
99:	learn: 1002.5047654	total: 5s	remaining: 0us
0:	learn: 1532.6295875	total: 81.4ms	remaining: 8.06s
1:	learn: 1396.0752938	total: 148ms	remaining: 7.24s
2:	learn: 1296.6968683	total: 154ms	remaining: 4.98s
3:	learn: 1230.3403224	total: 160ms	remaining: 3.85s
4:	learn: 1185.7334602	total: 170ms	remaining: 3.24s
5:	learn: 1150.2782675	total: 241ms	remaining: 3.78s
6:	learn: 1125.5521071	total: 326ms	remaining: 4.33s
7:	learn: 1111.3081581	total: 385ms	remaining: 4.43s
8:	learn: 1100.6200157	total: 450ms	remaining: 4.55s
9:	learn: 1093.3825440	total: 537ms	remaining: 4.83s
10:	learn: 1088.3199705	total: 601ms	remaining: 4.86s
11:	learn: 1083.8186566	total: 727ms	remaining: 5.33s
12:	learn: 1081.1949084	total: 796ms	remaining: 5.32s
13:	learn: 1078.8291647	total: 858ms	remaining: 5.27s
14:	learn: 1076.4427484	total: 901ms	remaining: 5.1s
15:	learn: 1075.6298489	total: 918ms	remai

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

49:	learn: 1028.7111678	total: 2.29s	remaining: 2.29s
50:	learn: 1028.2568621	total: 2.31s	remaining: 2.22s
51:	learn: 1027.3329031	total: 2.32s	remaining: 2.14s
52:	learn: 1026.4536479	total: 2.36s	remaining: 2.1s
53:	learn: 1025.6190399	total: 2.47s	remaining: 2.1s
54:	learn: 1024.6825100	total: 2.48s	remaining: 2.03s
55:	learn: 1024.2372465	total: 2.49s	remaining: 1.96s
56:	learn: 1023.8425249	total: 2.5s	remaining: 1.89s
57:	learn: 1023.0587087	total: 2.52s	remaining: 1.82s
58:	learn: 1021.9130620	total: 2.52s	remaining: 1.75s
59:	learn: 1021.1798432	total: 2.53s	remaining: 1.69s
60:	learn: 1020.5321072	total: 2.54s	remaining: 1.62s
61:	learn: 1019.3334004	total: 2.54s	remaining: 1.56s
62:	learn: 1018.7962338	total: 2.55s	remaining: 1.5s
63:	learn: 1018.4518397	total: 2.61s	remaining: 1.47s
64:	learn: 1018.1429631	total: 2.7s	remaining: 1.45s
65:	learn: 1017.0703697	total: 2.77s	remaining: 1.42s
66:	learn: 1016.3882772	total: 2.81s	remaining: 1.38s
67:	learn: 1015.5056990	total: 2.

47:	learn: 1033.9250333	total: 2.5s	remaining: 2.71s
48:	learn: 1033.3689911	total: 2.6s	remaining: 2.71s
49:	learn: 1032.5722174	total: 2.65s	remaining: 2.65s
50:	learn: 1031.8271252	total: 2.66s	remaining: 2.56s
51:	learn: 1030.8706885	total: 2.68s	remaining: 2.47s
52:	learn: 1030.1672070	total: 2.69s	remaining: 2.39s
53:	learn: 1029.6995209	total: 2.7s	remaining: 2.3s
54:	learn: 1028.4709647	total: 2.72s	remaining: 2.22s
55:	learn: 1027.7264779	total: 2.73s	remaining: 2.14s
56:	learn: 1027.0510484	total: 2.8s	remaining: 2.11s
57:	learn: 1026.1535495	total: 2.82s	remaining: 2.04s
58:	learn: 1025.4407464	total: 2.89s	remaining: 2.01s
59:	learn: 1025.0176795	total: 2.91s	remaining: 1.94s
60:	learn: 1024.4125734	total: 2.92s	remaining: 1.86s
61:	learn: 1024.0513209	total: 2.92s	remaining: 1.79s
62:	learn: 1023.5125339	total: 2.93s	remaining: 1.72s
63:	learn: 1022.9021218	total: 2.94s	remaining: 1.65s
64:	learn: 1022.5586916	total: 2.99s	remaining: 1.61s
65:	learn: 1021.8178034	total: 2.

KeyboardInterrupt: 

In [ ]:
gcv.best_score_,gcv.best_params_

In [11]:
test

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,FDW58,20.750,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1
1,FDW14,8.300,reg,0.038428,Dairy,87.3198,OUT017,2007,NaN,Tier 2,Supermarket Type1
2,NCN55,14.600,Low Fat,0.099575,Others,241.7538,OUT010,1998,NaN,Tier 3,Grocery Store
3,FDQ58,7.315,Low Fat,0.015388,Snack Foods,155.0340,OUT017,2007,NaN,Tier 2,Supermarket Type1
4,FDY38,NaN,Regular,0.118599,Dairy,234.2300,OUT027,1985,Medium,Tier 3,Supermarket Type3
...,...,...,...,...,...,...,...,...,...,...,...
5676,FDB58,10.500,Regular,0.013496,Snack Foods,141.3154,OUT046,1997,Small,Tier 1,Supermarket Type1
5677,FDD47,7.600,Regular,0.142991,Starchy Foods,169.1448,OUT018,2009,Medium,Tier 3,Supermarket Type2
5678,NCO17,10.000,Low Fat,0.073529,Health and Hygiene,118.7440,OUT045,2002,NaN,Tier 2,Supermarket Type1
5679,FDJ26,15.300,Regular,0.000000,Canned,214.6218,OUT017,2007,NaN,Tier 2,Supermarket Type1


In [12]:
bm = gcv.best_estimator_
y_pred = bm.predict(test)
y_pred

array([1648.2816, 1438.0696,  679.7606, ..., 1929.1388, 3639.1345,
       1291.0635], dtype=float32)

In [15]:
submission = pd.read_csv("sample_submission_8RXa3c6.csv")
submission["Item_Outlet_Sales"] = y_pred

In [16]:
submission.to_csv("AnalyticsVidhyaSubmission.csv", index = False)